# Phonon dispersion (DFT) — siman + phonopy

Finite-displacement phonon bands for the lowest-Gibbs DFT cells:

- **layered P2₁/c** — already a 2×2×2 supercell (`layered_P2_1_c.sc`)
- **diaspore α** (Pnma) — already a 4×3×1 supercell (`diaspore_alpha.su`)

Both come from `phonons_setup_eos` version 100 (EOS-fitted volume). Every lattice vector is longer than 10 Å, so **DIM = 1 1 1** (no extra supercell). Displacements are generated on that cell; `primitive_matrix='auto'` keeps band paths in the primitive BZ.

1. Write the relaxed `.end` structures from the siman database
2. `phonopy` displacements → one zen4 job **per displacement** under `phonons_dispersion/{phase}/{N:03d}/`
3. Collect finished OUTCARs, build `FORCE_SETS`, plot in-plane + out-of-plane bands

Flags: `LAUNCH_JOBS` submits; `READ_AND_PROCESS` reads finished jobs and runs phonopy.


In [ ]:
%%capture
import os
import glob
import shutil
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import siman
from siman.calc_manage import smart_structure_read, add, res
from siman import header
from siman.database import write_database, read_database
from siman.set_functions import read_vasp_sets
from siman.header import db, _update_configuration
from pydoc import importfile

_update_configuration("project_conf.py")
read_database()
project_sets = importfile("project_sets.py")
varset = read_vasp_sets(project_sets.user_vasp_sets, override_global=1)

header.PATH2EDITOR = "notepad.exe"
header.PATH2POTENTIALS = "/home/a.burov/soft/vasp_potentials/potpaw_PBE_MPIE/"


In [ ]:
from ase.io import read, write
from ase import Atoms
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import write_FORCE_SETS, parse_FORCE_SETS
from phonopy.interface.vasp import parse_set_of_forces, write_vasp

try:
    from phonopy import load as phonopy_load
except ImportError:
    try:
        from phonopy.cui.load import load as phonopy_load
    except ImportError:
        phonopy_load = None


In [ ]:
CLUSTER_ROOT = Path("/home/a.burov/icys_2025/niohf")
LOCAL_ROOT = Path("/home/arseniy/Desktop/work/niohf")
ROOT = CLUSTER_ROOT if CLUSTER_ROOT.exists() else LOCAL_ROOT

DISP_ROOT = ROOT / "phonons_dispersion"
FIGURES_ROOT = DISP_ROOT / "figures"
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)

VASP_SET = "phonons_disp"
CLUSTER = "zen4"
CALC_VER = 1  # one version per folder

# True: phonopy -d + siman add()  (existing db entries are skipped)
LAUNCH_JOBS = True
# True: res() finished jobs and run phonopy when the set is complete
READ_AND_PROCESS = True
SKIP_EXISTING = True

print(f"ROOT = {ROOT}")
print(f"set  = {VASP_SET}   cluster = {CLUSTER}")


In [ ]:
# Already-relaxed EOS supercells (lattice vectors all > 10 A) → DIM = 1 1 1.
# Band paths are for the primitive cell (phonopy primitive_matrix='auto').
# layered P21/c: Z=(0,1/2,0) is the unique-b / stacking-related direction.
# diaspore alpha (Pnma): Z=(0,0,1/2) is along the long c axis.

IDENTITY = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]

PHASES = {
    "layered_P2_1_c": {
        "db_key": ("layered_P2_1_c.sc", "phonons_setup_eos", 100),
        "structure": ROOT / "optimized_best/dft/layered_P2_1_c.vasp",
        "dim": IDENTITY,
        "spacegroup": "P2_1/c",
        "paths": {
            "out_of_plane": {
                "labels": [r"$\Gamma$", "Z", "D", "B", r"$\Gamma$"],
                "qpoints": [
                    [0.0, 0.0, 0.0],
                    [0.0, 0.5, 0.0],
                    [0.0, 0.5, 0.5],
                    [0.0, 0.0, 0.5],
                    [0.0, 0.0, 0.0],
                ],
            },
            "in_plane": {
                "labels": [r"$\Gamma$", "A", "E", "Z", r"C$_2$", r"Y$_2$", r"$\Gamma$"],
                "qpoints": [
                    [0.0, 0.0, 0.0],
                    [-0.5, 0.0, 0.5],
                    [-0.5, 0.5, 0.5],
                    [0.0, 0.5, 0.0],
                    [-0.5, 0.5, 0.0],
                    [-0.5, 0.0, 0.0],
                    [0.0, 0.0, 0.0],
                ],
            },
        },
    },
    "diaspore_alpha": {
        "db_key": ("diaspore_alpha.su", "phonons_setup_eos", 100),
        "structure": ROOT / "optimized_best/dft/diaspore_alpha.vasp",
        "dim": IDENTITY,
        "spacegroup": "Pnma",
        "paths": {
            "in_plane": {
                "labels": [r"$\Gamma$", "X", "S", "Y", r"$\Gamma$"],
                "qpoints": [
                    [0.0, 0.0, 0.0],
                    [0.5, 0.0, 0.0],
                    [0.5, 0.5, 0.0],
                    [0.0, 0.5, 0.0],
                    [0.0, 0.0, 0.0],
                ],
            },
            "out_of_plane": {
                "labels": [r"$\Gamma$", "Z", "U", "R", "T", "Z"],
                "qpoints": [
                    [0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.5],
                    [0.5, 0.0, 0.5],
                    [0.5, 0.5, 0.5],
                    [0.0, 0.5, 0.5],
                    [0.0, 0.0, 0.5],
                ],
            },
        },
        "vertical": {
            "labels": ["X", "U", "|", "Y", "T", "|", "S", "R"],
            "qpoints": [
                [0.5, 0.0, 0.0],
                [0.5, 0.0, 0.5],
                None,
                [0.0, 0.5, 0.0],
                [0.0, 0.5, 0.5],
                None,
                [0.5, 0.5, 0.0],
                [0.5, 0.5, 0.5],
            ],
        },
    },
}


In [ ]:
def phase_dir(phase):
    p = DISP_ROOT / phase
    p.mkdir(parents=True, exist_ok=True)
    return p


def job_name(phase, i):
    """Unique siman name so each displacement gets its own folder and sbatch."""
    return f"disp_{phase}_{i:03d}"


def it_folder_for(phase, i):
    return f"phonons_dispersion/{phase}/{i:03d}"


def db_key(phase, i):
    return (job_name(phase, i), VASP_SET, CALC_VER)


def ase_to_phonopy(atoms):
    return PhonopyAtoms(
        symbols=atoms.get_chemical_symbols(),
        cell=atoms.get_cell().array,
        scaled_positions=atoms.get_scaled_positions(),
    )


def phonopy_to_ase(ph_atoms):
    return Atoms(
        symbols=ph_atoms.symbols,
        cell=ph_atoms.cell,
        scaled_positions=ph_atoms.scaled_positions,
        pbc=True,
    )


def write_relaxed_from_db(phase=None):
    """Write phonons_setup_eos v100 .end supercells. Do not reduce or replicate."""
    phases = [phase] if phase else list(PHASES)
    written = []
    for name in phases:
        cfg = PHASES[name]
        key = cfg["db_key"]
        st = db[key].copy().end
        dest = Path(cfg["structure"])
        dest.parent.mkdir(parents=True, exist_ok=True)
        st.write_poscar(str(dest))
        lengths = np.linalg.norm(np.array(st.rprimd), axis=1)
        maxf = getattr(db[key], "maxforce", None)
        print(
            f"{name}: {len(st.xred)} atoms  abc={lengths} Å  "
            f"maxforce={maxf} meV/A  → {dest}"
        )
        if np.min(lengths) < 10.0:
            raise ValueError(f"{name}: a lattice vector is < 10 Å: {lengths}")
        written.append(dest)
    return written


def path_segments(phase, phase_cfg, include_vertical=True):
    """Return [(name, labels, qpoints), ...] in plot order."""
    order = {
        "layered_P2_1_c": ["out_of_plane", "in_plane"],
        "diaspore_alpha": ["in_plane", "out_of_plane"],
    }[phase]
    segs = []
    for key in order:
        p = phase_cfg["paths"][key]
        segs.append((key, p["labels"], np.array(p["qpoints"], dtype=float)))
    if include_vertical and "vertical" in phase_cfg:
        raw_l = phase_cfg["vertical"]["labels"]
        raw_q = phase_cfg["vertical"]["qpoints"]
        cur_l, cur_q = [], []
        n = 0
        for lab, q in zip(raw_l, raw_q):
            if q is None or lab == "|":
                if cur_q:
                    n += 1
                    segs.append((f"vertical_{n}", cur_l, np.array(cur_q, dtype=float)))
                cur_l, cur_q = [], []
                continue
            cur_l.append(lab)
            cur_q.append(q)
        if cur_q:
            n += 1
            segs.append((f"vertical_{n}", cur_l, np.array(cur_q, dtype=float)))
    return segs


def write_band_conf(phase, phase_cfg, dest):
    dim = phase_cfg["dim"]
    lines = [
        "ATOM_NAME = Ni H O F",
        f"DIM = {dim[0][0]} {dim[1][1]} {dim[2][2]}",
        "PRIMITIVE_AXES = AUTO",
        "BAND_POINTS = 51",
    ]
    all_labels = []
    for name, labels, qs in path_segments(phase, phase_cfg):
        flat = "  ".join(f"{x:.6f} {y:.6f} {z:.6f}" for x, y, z in qs)
        lines.append(f"# {name}")
        lines.append(f"BAND = {flat}")
        if all_labels:
            all_labels.append("|")
        all_labels.extend(labels)
    token = []
    for lab in all_labels:
        token.append("G" if lab in (r"$\Gamma$", "G") else lab.replace("$", "").replace("\\", ""))
    lines.append("BAND_LABELS = " + " ".join(token))
    dest.write_text("\n".join(lines) + "\n")
    return dest


write_relaxed_from_db()


In [ ]:
def generate_displacements(phase, distance=0.01):
    cfg = PHASES[phase]
    work = phase_dir(phase)
    write_relaxed_from_db(phase)
    atoms = read(str(cfg["structure"]))
    write(str(work / "POSCAR"), atoms, format="vasp", direct=True)

    # Cell is already the 2x2x2 / 4x3x1 supercell; do not seekpath-reduce it.
    phonon = Phonopy(
        ase_to_phonopy(atoms),
        supercell_matrix=cfg["dim"],
        primitive_matrix="auto",
    )
    phonon.generate_displacements(distance=distance)
    phonon.save(str(work / "phonopy_disp.yaml"), settings={"force_sets": False})

    sposcar = phonon.supercell
    write_vasp(str(work / "SPOSCAR"), sposcar)
    disp_cells = phonon.supercells_with_displacements
    n_disp = 0
    for i, sc in enumerate(disp_cells, start=1):
        if sc is None:
            continue
        write_vasp(str(work / f"POSCAR-{i:03d}"), sc)
        n_disp += 1

    write_band_conf(phase, cfg, work / "band.conf")
    print(f"{phase}: {n_disp} displacements, dim={cfg['dim'][0][0]}x{cfg['dim'][1][1]}x{cfg['dim'][2][2]}")
    print(f"  work dir: {work}")
    return n_disp


def list_displacement_ids(phase):
    work = phase_dir(phase)
    ids = []
    for p in work.glob("POSCAR-*"):
        ids.append(int(p.name.split("-")[-1]))
    return sorted(ids)


## Generate displacements and launch with siman

Each `POSCAR-00N` is a **separate** siman job and folder:

`phonons_dispersion/{phase}/{N:03d}/disp_{phase}_{N:03d}.phonons_disp/`

zen4, `srun vasp_gam`, Gamma-only `ngkpt=[1,1,1]`, 24 h and 64 GB per displacement.


In [ ]:
def launch_phase(phase, run=2):
    generate_displacements(phase)
    work = phase_dir(phase)
    ids = list_displacement_ids(phase)
    print(f"\nLaunching {phase}: {len(ids)} separate zen4 jobs  (run={run})")
    for i in ids:
        key = db_key(phase, i)
        if SKIP_EXISTING and key in db:
            print(f"  skip {i} (already in db as {key[0]})")
            continue
        poscar = work / f"POSCAR-{i:03d}"
        st = smart_structure_read(str(poscar))
        add(
            job_name(phase, i),
            VASP_SET,
            CALC_VER,
            it_folder=it_folder_for(phase, i),
            input_st=st,
            ngkpt=[1, 1, 1],
            cluster=CLUSTER,
            up="up2",
            run=run,
        )
    return ids


if LAUNCH_JOBS:
    for phase in PHASES:
        launch_phase(phase, run=2)
    write_database()
    print("\nDatabase updated. Jobs submitted (or inputs written).")
else:
    for phase in PHASES:
        generate_displacements(phase)
    print("LAUNCH_JOBS=False — displacements only, no siman add()")


## Read results and build dispersions

Run this cell after the queue finishes. Incomplete displacements are reported and skipped.


In [ ]:
def _looks_finished(outcar: Path) -> bool:
    if not outcar.exists() or "unfinished" in outcar.name:
        return False
    try:
        tail = outcar.read_text(errors="ignore")[-8000:]
    except Exception:
        return outcar.stat().st_size > 50000
    return ("General timing" in tail) or ("Voluntary number of scf" in tail)


def outcar_path(phase, i):
    """Finished OUTCAR for displacement i (new per-folder layout or db path)."""
    candidates = []
    key = db_key(phase, i)
    if key in db:
        calc = db[key]
        path = calc.path.get("output") if hasattr(calc, "path") else None
        if path:
            candidates.append(Path(path))
            candidates.append(ROOT / path)
            candidates.append(Path(str(path).replace("OUTCAR_unfinished", "OUTCAR")))

    folder = ROOT / it_folder_for(phase, i)
    candidates.extend(folder.glob("**/1.OUTCAR"))
    candidates.extend(folder.glob("**/*.OUTCAR"))

    seen = set()
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if p in seen or not p.is_file():
            continue
        seen.add(p)
        if _looks_finished(p):
            return p
    return None


def read_phase(phase):
    ids = list_displacement_ids(phase)
    done, missing = [], []
    for i in ids:
        try:
            res(job_name(phase, i), VASP_SET, CALC_VER, up="up2", cluster=CLUSTER)
        except Exception as exc:
            print(f"  res() {phase} {i}: {exc}")
        if outcar_path(phase, i) is not None:
            done.append(i)
        else:
            missing.append(i)
    print(f"{phase}: {len(done)} finished, {len(missing)} missing {missing[:12]}")
    return done, missing


def collect_force_files(phase, done_ids):
    """Copy finished OUTCAR/vasprun into the phase dir as 001.OUTCAR for phonopy -f."""
    work = phase_dir(phase)
    files = []
    for i in done_ids:
        oc = outcar_path(phase, i)
        if oc is None:
            continue
        dest = work / f"{i:03d}.OUTCAR"
        if oc.resolve() != dest.resolve():
            shutil.copy2(oc, dest)
        for cand in (
            oc.with_name(oc.name.replace("OUTCAR", "vasprun.xml")),
            oc.parent / "vasprun.xml",
            oc.parent / "1.vasprun.xml",
        ):
            if cand.exists() and "unfinished" not in cand.name:
                vdest = work / f"{i:03d}.vasprun.xml"
                if cand.resolve() != vdest.resolve():
                    shutil.copy2(cand, vdest)
                files.append(vdest)
                break
        else:
            files.append(dest)
    return files


def run_phonopy_forces(phase, force_files):
    work = phase_dir(phase)
    yaml = work / "phonopy_disp.yaml"
    if not yaml.exists():
        raise FileNotFoundError(f"missing {yaml}; regenerate displacements first")

    phonopy_bin = shutil.which("phonopy")
    if phonopy_bin:
        cmd = [phonopy_bin, "-f"] + [str(p.name) for p in force_files]
        result = subprocess.run(cmd, cwd=work, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(f"phonopy -f failed for {phase}")
        return work / "FORCE_SETS"

    n_atoms = len(read(str(work / "SPOSCAR")))
    forces = parse_set_of_forces(n_atoms, [str(p) for p in force_files])
    write_FORCE_SETS(forces, filename=str(work / "FORCE_SETS"))
    return work / "FORCE_SETS"


def build_band_path(phase, phase_cfg, npoints=51):
    bands, labels, path_connections = [], [], []
    for _, seg_labels, qs in path_segments(phase, phase_cfg):
        nseg = len(qs) - 1
        for j in range(nseg):
            bands.append(np.linspace(qs[j], qs[j + 1], npoints))
        path_connections.extend([True] * (nseg - 1) + [False])
        labels.extend(seg_labels)
    return bands, labels, path_connections


def _band_xticks(distances, path_connections, labels):
    xticks, xlabels = [], []
    lab_i = 0
    for i, dist in enumerate(distances):
        new_group = i == 0 or not path_connections[i - 1]
        if new_group:
            xticks.append(float(dist[0]))
            xlabels.append(labels[lab_i] if lab_i < len(labels) else "")
            lab_i += 1
        xticks.append(float(dist[-1]))
        xlabels.append(labels[lab_i] if lab_i < len(labels) else "")
        lab_i += 1
    return xticks, xlabels


def _freq_break(frequencies, gap_min=8.0):
    """Find a large empty gap between lattice modes and O–H stretches."""
    vals = np.concatenate([np.asarray(f).ravel() for f in frequencies])
    vals = vals[np.isfinite(vals)]
    if vals.size < 10:
        return None
    lo, hi = float(np.min(vals)), float(np.max(vals))
    # ignore deep imaginary modes for gap search
    pos = np.sort(vals[vals > -5.0])
    if pos.size < 10:
        return None
    d = np.diff(pos)
    i = int(np.argmax(d))
    if d[i] < gap_min:
        return None
    mid_lo = float(pos[i])
    mid_hi = float(pos[i + 1])
    # require the upper block to be a thin stretch band near the top
    if mid_hi - mid_lo < gap_min:
        return None
    return (lo, mid_lo, mid_hi, hi)


def _style_band_ax(ax, xticks, xlabels, ylabel=None, show_x=True):
    ax.axhline(0.0, color="#9aa3ad", lw=0.7, ls="--", zorder=0)
    for x in xticks:
        ax.axvline(x, color="#d9dee5", lw=0.6, zorder=0)
    ax.set_xlim(xticks[0], xticks[-1])
    ax.tick_params(axis="both", labelsize=11, length=3.5, width=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=12)
    if show_x:
        ax.set_xticks(xticks)
        ax.set_xticklabels(xlabels, fontsize=12)
    else:
        ax.set_xticks(xticks)
        ax.set_xticklabels([])


def plot_and_save_bands(phase, phonon):
    """Publication-style bands with an optional broken y-axis across the O–H gap."""
    FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
    plt.close("all")
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
        "axes.linewidth": 1.0,
        "figure.dpi": 120,
    })

    cfg = PHASES[phase]
    bands, labels, path_connections = build_band_path(phase, cfg)
    color = "#1f4e79"
    lw = 0.95

    try:
        data = phonon.get_band_structure_dict()
        distances = data["distances"]
        frequencies = data["frequencies"]
    except Exception as exc:
        print(f"  custom extract failed ({exc}); phonopy default plot")
        phonon.plot_band_structure()
        fig = plt.gcf()
        fig.set_size_inches(10, 5.2)
        out_png = FIGURES_ROOT / f"dft_{phase}_bands.png"
        out_pdf = FIGURES_ROOT / f"dft_{phase}_bands.pdf"
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        fig.savefig(out_pdf, bbox_inches="tight")
        plt.close(fig)
        print(f"  saved {out_png}")
        return

    xticks, xlabels = _band_xticks(distances, path_connections, labels)
    brk = _freq_break(frequencies)

    if brk is None:
        fig, ax = plt.subplots(figsize=(10.5, 5.4))
        for dist, freq in zip(distances, frequencies):
            ax.plot(dist, freq, color=color, lw=lw)
        all_f = np.concatenate([np.asarray(f).ravel() for f in frequencies])
        ymin = min(-2.0, float(np.nanmin(all_f)) - 1.0)
        ymax = float(np.nanmax(all_f)) * 1.04
        ax.set_ylim(ymin, ymax)
        _style_band_ax(ax, xticks, xlabels, ylabel="Frequency (THz)")
        ax.set_title(f"DFT — {phase}", fontsize=13, pad=8)
    else:
        lo, mid_lo, mid_hi, hi = brk
        # pad the two visible windows
        y_lo_max = mid_lo + 0.08 * max(mid_lo - lo, 5.0)
        y_hi_min = mid_hi - 0.08 * max(hi - mid_hi, 2.0)
        y_hi_max = hi + 0.04 * max(hi - mid_hi, 2.0)
        # keep a little room for soft imaginary modes
        y_lo_min = min(-2.0, lo - 1.0)

        fig, (ax_hi, ax_lo) = plt.subplots(
            2,
            1,
            sharex=True,
            figsize=(10.5, 6.2),
            gridspec_kw={"height_ratios": [1.0, 3.2], "hspace": 0.05},
        )
        for ax in (ax_hi, ax_lo):
            for dist, freq in zip(distances, frequencies):
                ax.plot(dist, freq, color=color, lw=lw)

        ax_hi.set_ylim(y_hi_min, y_hi_max)
        ax_lo.set_ylim(y_lo_min, y_lo_max)
        _style_band_ax(ax_hi, xticks, xlabels, show_x=False)
        _style_band_ax(ax_lo, xticks, xlabels, ylabel="Frequency (THz)")
        ax_hi.set_title(f"DFT — {phase}", fontsize=13, pad=8)

        # diagonal break marks
        d = 0.012
        kwargs = dict(transform=ax_hi.transAxes, color="0.35", clip_on=False, lw=0.9)
        ax_hi.plot((-d, +d), (-d, +d), **kwargs)
        ax_hi.plot((1 - d, 1 + d), (-d, +d), **kwargs)
        kwargs.update(transform=ax_lo.transAxes)
        ax_lo.plot((-d, +d), (1 - d, 1 + d), **kwargs)
        ax_lo.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)
        ax_hi.spines["bottom"].set_visible(False)
        ax_lo.spines["top"].set_visible(False)
        ax_hi.tick_params(bottom=False)

    out_png = FIGURES_ROOT / f"dft_{phase}_bands.png"
    out_pdf = FIGURES_ROOT / f"dft_{phase}_bands.pdf"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out_png}" + ("  [broken y-axis]" if brk else ""))


def load_phonon(work):
    """Load FORCE_SETS + phonopy_disp.yaml on both old and new phonopy."""
    yaml = str(work / "phonopy_disp.yaml")
    fsets = str(work / "FORCE_SETS")
    load_fn = phonopy_load or getattr(Phonopy, "load", None)
    if load_fn is not None:
        for kwargs in (
            {"phonopy_yaml": yaml, "force_sets_filename": fsets},
            {"phonopy_yaml": yaml, "force_sets_file_name": fsets},
        ):
            try:
                return load_fn(**kwargs)
            except TypeError:
                continue

    from phonopy.interface.phonopy_yaml import PhonopyYaml

    phy = PhonopyYaml()
    phy.read(yaml)
    phonon = Phonopy(
        phy.unitcell,
        supercell_matrix=phy.supercell_matrix,
        primitive_matrix=phy.primitive_matrix,
    )
    if getattr(phy, "dataset", None) is not None:
        phonon.dataset = phy.dataset
    force_sets = parse_FORCE_SETS(filename=fsets)
    if hasattr(phonon, "set_displacement_dataset") and isinstance(force_sets, dict):
        phonon.set_displacement_dataset(force_sets)
    elif hasattr(phonon, "set_forces"):
        first = force_sets["first_atoms"] if isinstance(force_sets, dict) else force_sets
        phonon.set_forces([d["forces"] for d in first])
    else:
        phonon.dataset = force_sets
    if hasattr(phonon, "produce_force_constants"):
        phonon.produce_force_constants()
    else:
        phonon.get_force_constants()
    return phonon


def run_bands(phonon, bands, labels, path_connections):
    try:
        phonon.run_band_structure(
            bands,
            is_band_connection=True,
            path_connections=path_connections,
            labels=labels,
        )
        return
    except (TypeError, AttributeError):
        pass
    try:
        phonon.set_band_structure(bands, is_band_connection=True, labels=labels)
    except TypeError:
        phonon.set_band_structure(bands)


def process_phase(phase, done_ids):
    cfg = PHASES[phase]
    work = phase_dir(phase)
    force_files = collect_force_files(phase, done_ids)
    if len(force_files) < 3:
        print(f"{phase}: only {len(force_files)} force files — skip phonopy")
        return False

    expected = list_displacement_ids(phase)
    if expected and len(force_files) != len(expected):
        print(
            f"{phase}: {len(force_files)}/{len(expected)} forces; "
            "FORCE_SETS needs the full set in order — skip until complete"
        )
        return False

    print(f"\n=== phonopy {phase} ({len(force_files)} files) ===")
    run_phonopy_forces(phase, force_files)

    phonon = load_phonon(work)
    bands, labels, path_connections = build_band_path(phase, cfg)
    run_bands(phonon, bands, labels, path_connections)
    try:
        phonon.write_yaml_band_structure(filename=str(work / "band.yaml"))
    except TypeError:
        phonon.write_yaml_band_structure()
        src = Path("band.yaml")
        if src.exists():
            src.replace(work / "band.yaml")
    plot_and_save_bands(phase, phonon)
    return True


In [ ]:
if READ_AND_PROCESS:
    results = {}
    for phase in PHASES:
        # regenerate yaml/POSCAR/band.conf if this tree was copied without them
        if not (phase_dir(phase) / "phonopy_disp.yaml").exists():
            generate_displacements(phase)
        done, missing = read_phase(phase)
        results[phase] = {"done": done, "missing": missing}
        if missing:
            print(f"  wait for jobs disp_{phase}_XXX / {VASP_SET} / {missing}")
            continue
        process_phase(phase, done)
    write_database()
else:
    print("READ_AND_PROCESS=False")


## Notes

- Structures: `db["layered_P2_1_c.sc", "phonons_setup_eos", 100].end` (P2₁/c, 2×2×2) and `db["diaspore_alpha.su", "phonons_setup_eos", 100].end` (Pnma, 4×3×1). Written to `optimized_best/dft/`.
- No extra supercell: all lattice vectors are already > 10 Å, `DIM = 1 1 1`, `primitive_matrix='auto'`
- **layered P2₁/c path:** out-of-plane $\Gamma$–Z–D–B–$\Gamma$, then in-plane $\Gamma$–A–E–Z–C₂–Y₂–$\Gamma$
- **diaspore α path:** in-plane $\Gamma$–X–S–Y–$\Gamma$, out-of-plane $\Gamma$–Z–U–R–T–Z, plus X–U, Y–T, S–R
- Each displacement is its own zen4 job:
  `phonons_dispersion/{phase}/{N:03d}/disp_{phase}_{N:03d}.phonons_disp/1.OUTCAR`
- Re-run the read cell until every folder has a finished OUTCAR; phonopy starts only then
- Set `LAUNCH_JOBS = False` after the first submit so you do not resubmit
- Old `layered_p2c` / `delta` jobs used a different cell; do not reuse those OUTCARs
